# 03. deque

**Tier:** Intermediate. **Why you'd use this:** A queue of jobs should pop from the front and append at the back. On a list, popping the front shifts every later item. A `deque` makes both ends cheap.

**After this lesson you can:** use `append` and `popleft` as a queue; rotate; bound a deque with `maxlen`.
About 25 minutes. **Uses:** [Lists](../04-collections/01-lists-lesson.ipynb)


## Quick review

1. Why can reading `defaultdict[key]` insert a key?
2. When do you prefer `Counter` over `defaultdict(int)`?

<details><summary>Answers</summary>

`__missing__` stores the factory result. Prefer `Counter` for frequency helpers like `most_common`.
</details>


## The idea

A `deque` (double-ended queue) is a sequence optimized for appends and pops at both ends. `append` and `popleft` implement a FIFO queue. `appendleft` and `pop` work at the other end. `maxlen` drops from the opposite end when the deque is full, which makes a fixed-size recent window.

<details><summary>Explain more</summary>

A list is fast at the right end and slow at the left because it is one block of pointers. A deque uses linked blocks so both ends stay cheap.
</details>


## See it work

Serve the front of a queue, then keep only the last few events.


In [1]:
from collections import deque


def serve(jobs):
    # 1. Pop from the left, append on the right elsewhere.
    queue = deque(jobs)
    return queue.popleft()


print(serve(["a", "b", "c"]))
recent = deque(maxlen=2)
for event in (1, 2, 3):
    recent.append(event)
print(list(recent))


a
[2, 3]


## Check yourself

Answer these before you run the cell.

1. Which call removes the front item of a deque?
2. After three appends into `deque(maxlen=2)`, how many items remain?


In [2]:
from collections import deque

queue = deque([1, 2, 3])
print(queue.popleft())
print(list(queue))
recent = deque(maxlen=2)
for event in (1, 2, 3):
    recent.append(event)
print(list(recent))


1
[2, 3]
[2, 3]


<details><summary>Answers</summary>

`popleft`. Two items remain: the oldest was dropped when the third arrived.
</details>


## Why it works this way

A list stores pointers in one contiguous block. Inserting or deleting at index `0` moves every later pointer, so front operations are O(n). A deque stores pointers in linked blocks. Adding or removing at either end updates a block edge without shifting the middle. That is why a queue should be a deque, not a list, once the workload is real. Deep dive: [How list works](../../internals/03-lists/01-dynamic-array-lesson.ipynb).

`timeit` makes the cost visible. Popping from the front of a growing list slows down as n grows. The same workload on a deque stays near flat. Random middle indexing is still a list strength because a deque may walk blocks to reach an interior index. A deque is not a general replacement for every list. Choose it when the hot path is the ends.

`maxlen` is a ring-buffer policy: once full, each new append discards from the other end. Memory stays fixed at about k items. Use it for "last k events" without writing the drop logic yourself. Rotation moves an item from one end to the other in constant time, which matches round-robin queues.

The design is a queue and stack toolkit on one type. Reach for `popleft` when order is first-in-first-out. Reach for `appendleft` when new work should jump the queue. Reach for a list when you need slices, sorts, or heavy middle access. Measuring both sides once prevents the classic "it worked on ten items" surprise in production.


In [3]:
import timeit
from collections import deque


def drain_list(n=3_000):
    items = list(range(n))
    while items:
        items.pop(0)


def drain_deque(n=3_000):
    items = deque(range(n))
    while items:
        items.popleft()


print(timeit.timeit(drain_list, number=20))
print(timeit.timeit(drain_deque, number=20))


0.012576875000377186
0.0015792909980518743


## When to use it

Use a deque for queues, rotating buffers, and maxlen windows. Use a list for indexed access and slices. Don't call `pop(0)` on a list in a hot loop when a deque will do.


## Common mistake

Building a queue with a list and `pop(0)`. It works on tiny inputs and quietly becomes expensive as the queue grows.


In [4]:
jobs = ["a", "b", "c"]
first = jobs.pop(0)
print(first, jobs)


a ['b', 'c']


Use `deque` and `popleft` so the front removal stays cheap.


In [5]:
from collections import deque

jobs = deque(["a", "b", "c"])
first = jobs.popleft()
print(first, list(jobs))


a ['b', 'c']


## Try it

Change `serve` so it rotates the front job to the back instead of removing it, and returns the new order as a list. Which two deque operations do that?


## Recap

- A deque is O(1) at both ends. A list is O(n) at the front.
- `popleft` and `append` make a queue.
- `maxlen` keeps a fixed window.

**Next:** [practice](03-deque-practice.ipynb)
